In [ ]:
!pip install ultralytics --upgrade -q

In [1]:
!which trtexec

In [4]:
!pip install tensorrt-cu12 tensorrt-cu12-bindings tensorrt-cu12-libs -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 2.7 MB/s eta 0:00:00


In [1]:
!pip install ultralytics --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.3 MB/s eta 0:00:00


In [5]:
import tensorrt as trt
print(trt.__version__)

11.0.0.114


In [6]:
import tensorrt as trt
print([x for x in dir(trt.BuilderFlag) if not x.startswith('_')])

['DEBUG', 'DIRECT_IO', 'DISABLE_COMPILATION_CACHE', 'DISABLE_TIMING_CACHE', 'DISTRIBUTIVE_INDEPENDENCE', 'EDITABLE_TIMING_CACHE', 'ERROR_ON_TIMING_CACHE_MISS', 'EXCLUDE_LEAN_RUNTIME', 'GPU_FALLBACK', 'MONITOR_MEMORY', 'REFIT', 'REFIT_IDENTICAL', 'REFIT_INDIVIDUAL', 'SAFETY_SCOPE', 'SPARSE_WEIGHTS', 'STRICT_NANS', 'STRIP_PLAN', 'TF32', 'VERSION_COMPATIBLE', 'WEIGHT_STREAMING', 'name', 'value']


In [8]:
import tensorrt as trt

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

print("Building TensorRT engine from ONNX...")

builder  = trt.Builder(TRT_LOGGER)
network  = builder.create_network()
config   = builder.create_builder_config()
# No FP16 flag needed — TRT 11 selects optimal precision automatically

parser = trt.OnnxParser(network, TRT_LOGGER)
with open('yolov8n.onnx', 'rb') as f:
    success = parser.parse(f.read())

if not success:
    for i in range(parser.num_errors):
        print(parser.get_error(i))
else:
    print(f"ONNX parsed — {network.num_layers} layers")

serialized = builder.build_serialized_network(network, config)
runtime    = trt.Runtime(TRT_LOGGER)
engine     = runtime.deserialize_cuda_engine(serialized)

print(f"Engine built — {engine.num_io_tensors} I/O tensors")

Building TensorRT engine from ONNX...
ONNX parsed — 299 layers
Engine built — 2 I/O tensors


In [9]:
import torch
import time

context = engine.create_execution_context()

# Allocate input/output tensors on GPU
input_name  = engine.get_tensor_name(0)
output_name = engine.get_tensor_name(1)

input_tensor  = torch.randn(1, 3, 640, 640, dtype=torch.float32, device='cuda')
output_tensor = torch.zeros(1, 84, 8400, dtype=torch.float32, device='cuda')

# Point TRT engine to GPU memory locations
context.set_tensor_address(input_name,  input_tensor.data_ptr())
context.set_tensor_address(output_name, output_tensor.data_ptr())

# Warmup — discard first 10 runs
stream = torch.cuda.Stream()
for _ in range(10):
    context.execute_async_v3(stream.cuda_stream)
torch.cuda.synchronize()

# Benchmark — 100 runs
times = []
for _ in range(100):
    start = time.perf_counter()
    context.execute_async_v3(stream.cuda_stream)
    torch.cuda.synchronize()
    end   = time.perf_counter()
    times.append((end - start) * 1000)

mean_ms = sum(times) / len(times)
std_ms  = (sum((t - mean_ms)**2 for t in times) / len(times)) ** 0.5

print(f"TensorRT on T4 GPU")
print(f"Mean latency : {mean_ms:.2f} ms")
print(f"Std deviation: {std_ms:.2f} ms")

TensorRT on T4 GPU
Mean latency : 4.00 ms
Std deviation: 1.07 ms
